## 1. import libraries and configure the project

Before collecting data, I import the Python libraries needed throughout this notebook.

- `requests` is used to send requests to the SEC EDGAR API.
- `json` is used to work with the JSON responses returned by the API.
- `time` is used to add short pauses between requests to avoid sending too many requests too quickly.
- `os` is used to create and manage folders where the raw data will be stored.

I also define the location where the raw API responses will be saved (`data/raw/`) and create a `User-Agent` header, which the SEC requires when accessing its API.

In [89]:
import requests
import json
import time

RAW_DIR = "../data/raw"

HEADERS = {"User-Agent": "ME204 Final Project - S.E.Yzusqui@lse.ac.uk"}

## 2. Retrieve the company ticker database

The SEC identifies each company using a unique **Central Index Key (CIK)**. Before requesting the data, I fetch the SEC's company ticker database.

The company ticker mapping was obtained from the SEC's official site sec.gov. This file links company tickers to their CIK, which is required to find companies financial data.

I inspect the first 3 entries to view the structure of the data and confirm that the request was successful. This information will be used later to obtain the correct CIK for each company included in the analysis.

In [102]:
url = "https://www.sec.gov/files/company_tickers.json"
response = requests.get(url, headers=HEADERS)
ticker_data = response.json()


get_cik = {}

for entry in ticker_data.values():

    ticker = entry["ticker"]

    cik = str(entry["cik_str"]).zfill(10)

    get_cik[ticker] = cik

get_cik["XOM"] = "0000034088"


In [103]:
list(ticker_data.items())[:3]

[('0', {'cik_str': 1045810, 'ticker': 'NVDA', 'title': 'NVIDIA CORP'}),
 ('1', {'cik_str': 320193, 'ticker': 'AAPL', 'title': 'Apple Inc.'}),
 ('2', {'cik_str': 1652044, 'ticker': 'GOOGL', 'title': 'Alphabet Inc.'})]

## 3. Select companies 

The project analyses three sectors: Technology, Healthcare, and Energy. I select three companies from each sector to compare.

Using the company ticker database collected previously, I create a mapping between each company's ticker symbol and its CIK number.

In [104]:
companies = {
    "MSFT": {"name": "Microsoft", "sector": "Technology"},
    "NVDA": {"name": "Nvidia", "sector": "Technology"},
    "AAPL": {"name": "Apple", "sector": "Technology"},
    "PFE": {"name": "Pfizer", "sector": "Healthcare"},
    "JNJ": {"name": "Johnson & Johnson", "sector": "Healthcare"},
    "SYK": {"name": "Stryker", "sector": "Healthcare"},
    "XOM": {"name": "ExxonMobil", "sector": "Energy"},
    "CVX": {"name": "Chevron", "sector": "Energy"},
    "DUK": {"name": "Duke Energy", "sector": "Energy"},
}

for ticker in companies:

    name = companies[ticker]["name"]
    sector = companies[ticker]["sector"]
    cik = get_cik.get(ticker)

    print(ticker, "|", name, "|", sector, "|", cik)

MSFT | Microsoft | Technology | 0000789019
NVDA | Nvidia | Technology | 0001045810
AAPL | Apple | Technology | 0000320193
PFE | Pfizer | Healthcare | 0000078003
JNJ | Johnson & Johnson | Healthcare | 0000200406
SYK | Stryker | Healthcare | 0000310764
XOM | ExxonMobil | Energy | 0000034088
CVX | Chevron | Energy | 0000093410
DUK | Duke Energy | Energy | 0001326160


In [107]:
tags = [
    "Revenues",
    "ResearchAndDevelopmentExpense",
    "NetIncomeLoss",
    "NetCashProvidedByUsedInOperatingActivities",
    "PaymentsToAcquirePropertyPlantAndEquipment"
]

# Three companies file these concepts under different names.
tag_alternates = {
    ("PFE", "ResearchAndDevelopmentExpense"): "ResearchAndDevelopmentExpenseExcludingAcquiredInProcessCost",
    ("CVX", "PaymentsToAcquirePropertyPlantAndEquipment"): "PaymentsToAcquireProductiveAssets",
}

for ticker in companies:
    cik = get_cik.get(ticker)

    for tag in tags:
        fetch_tag = tag_alternates.get((ticker, tag), tag)
        url = f"https://data.sec.gov/api/xbrl/companyconcept/CIK{cik}/us-gaap/{fetch_tag}.json"
        response = requests.get(url, headers=HEADERS)

        if response.status_code == 200:
            data = response.json()
            filename = f"{RAW_DIR}/{ticker}_{tag}.json"
            with open(filename, "w") as f:
                json.dump(data, f)
            note = f" (via {fetch_tag})" if fetch_tag != tag else ""
            print(f"Saved: {ticker} - {tag}{note}")
        else:
            print(f"MISSING: {ticker} - {tag} (status {response.status_code})")

        time.sleep(0.2)

Saved: MSFT - Revenues
Saved: MSFT - ResearchAndDevelopmentExpense
Saved: MSFT - NetIncomeLoss
Saved: MSFT - NetCashProvidedByUsedInOperatingActivities
Saved: MSFT - PaymentsToAcquirePropertyPlantAndEquipment
Saved: NVDA - Revenues
Saved: NVDA - ResearchAndDevelopmentExpense
Saved: NVDA - NetIncomeLoss
Saved: NVDA - NetCashProvidedByUsedInOperatingActivities
Saved: NVDA - PaymentsToAcquirePropertyPlantAndEquipment
Saved: AAPL - Revenues
Saved: AAPL - ResearchAndDevelopmentExpense
Saved: AAPL - NetIncomeLoss
Saved: AAPL - NetCashProvidedByUsedInOperatingActivities
Saved: AAPL - PaymentsToAcquirePropertyPlantAndEquipment
Saved: PFE - Revenues
Saved: PFE - ResearchAndDevelopmentExpense (via ResearchAndDevelopmentExpenseExcludingAcquiredInProcessCost)
Saved: PFE - NetIncomeLoss
Saved: PFE - NetCashProvidedByUsedInOperatingActivities
Saved: PFE - PaymentsToAcquirePropertyPlantAndEquipment
Saved: JNJ - Revenues
Saved: JNJ - ResearchAndDevelopmentExpense
Saved: JNJ - NetIncomeLoss
Saved: JNJ 

In [87]:
check = ["MSFT","NVDA","AAPL","PFE","JNJ","SYK","XOM","CVX","DUK"]

for tkr in check:
    cik = "0000034088" if tkr == "XOM" else ticker_to_cik[tkr]
    d = requests.get(f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json", headers=HEADERS).json()
    gaap = d["facts"].get("us-gaap", {})
    print(f"{tkr:5} {d['entityName'][:26]:28} missing: {[t for t in tags if t not in gaap]}")
    time.sleep(1)

MSFT  MICROSOFT CORPORATION        missing: []
NVDA  NVIDIA CORP                  missing: []
AAPL  Apple Inc.                   missing: []
PFE   Pfizer Inc.                  missing: ['ResearchAndDevelopmentExpense']
JNJ   Johnson & Johnson            missing: []
SYK   STRYKER CORP                 missing: []
XOM   Exxon Mobil Corporation      missing: []
CVX   Chevron Corp                 missing: ['PaymentsToAcquirePropertyPlantAndEquipment']
DUK   DUKE ENERGY CORPORATION      missing: ['ResearchAndDevelopmentExpense']


In [69]:
cik = get_cik["PFE"]
url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json"
response = requests.get(url, headers=HEADERS)
facts = response.json()

all_tags = facts["facts"]["us-gaap"].keys()

matches = [tag for tag in all_tags if "Research" in tag or "OperatingIncome" in tag]
matches

['EffectiveIncomeTaxRateReconciliationNondeductibleExpenseResearchAndDevelopment',
 'EffectiveIncomeTaxRateReconciliationTaxCreditsResearch',
 'OtherOperatingIncomeExpenseNet',
 'ResearchAndDevelopmentExpenseExcludingAcquiredInProcessCost',
 'ResearchAndDevelopmentInProcess',
 'ResearchAndDevelopmentAssetAcquiredOtherThanThroughBusinessCombinationWrittenOff',
 'DeferredTaxAssetsInProcessResearchAndDevelopment',
 'IncomeTaxReconciliationTaxCreditsResearch']

In [70]:
tag_alternates = {
    "ResearchAndDevelopmentExpense": [
        "ResearchAndDevelopmentExpenseExcludingAcquiredInProcessCost"
    ],
    # we'll add more here as we discover them (e.g. for GrossProfit, OperatingIncomeLoss, capex)
}


In [71]:
company_facts = {}

for ticker in companies:
    cik = get_cik[ticker]
    url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json"
    response = requests.get(url, headers=HEADERS)

    if response.status_code != 200:
        print(f"{ticker}: request failed (status {response.status_code})")
        time.sleep(1)
        continue

    facts = response.json()

    if "facts" not in facts or "us-gaap" not in facts["facts"]:
        print(f"{ticker}: response has no us-gaap facts, keys = {list(facts.keys())}")
        time.sleep(1)
        continue

    company_facts[ticker] = facts
    available = facts["facts"]["us-gaap"].keys()

    for tag in tags:
        if tag in available:
            continue
        candidates = [
            t for t in available
            if any(kw.lower() in t.lower() for kw in tag_keywords[tag])
        ]
        print(f"\n{ticker} is missing {tag}")
        for c in candidates[:8]:
            print("   ->", c)

    time.sleep(1)


PFE is missing ResearchAndDevelopmentExpense
   -> EffectiveIncomeTaxRateReconciliationNondeductibleExpenseResearchAndDevelopment
   -> ResearchAndDevelopmentExpenseExcludingAcquiredInProcessCost
   -> ResearchAndDevelopmentInProcess
   -> ResearchAndDevelopmentAssetAcquiredOtherThanThroughBusinessCombinationWrittenOff
   -> DeferredTaxAssetsInProcessResearchAndDevelopment
XOM: response has no us-gaap facts, keys = ['cik', 'entityName', 'facts']

CVX is missing PaymentsToAcquirePropertyPlantAndEquipment
   -> AdditionalPaidInCapitalCommonStock
   -> CapitalizedCostsUncompletedWellsEquipmentAndFacilities
   -> CapitalizedExploratoryWellCostAdditionsPendingDeterminationOfProvedReserves
   -> CapitalizedExploratoryWellCostChargedToExpense
   -> CapitalizedExploratoryWellCostChargedToExpense1
   -> CapitalizedExploratoryWellCostPeriodIncreaseDecrease
   -> CapitalizedExploratoryWellCosts
   -> CapitalizedExploratoryWellCostsThatHaveBeenCapitalizedForPeriodGreaterThanOneYear

NEE is missin

In [72]:
xom = company_facts.get("XOM")
if xom is None:
    cik = get_cik["XOM"]
    xom = requests.get(f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json", headers=HEADERS).json()

print(xom["entityName"])
print(list(xom["facts"].keys()))

EXXON MOBIL CORP
['ffd']


In [73]:
import pandas as pd

rows = []
for ticker, facts in company_facts.items():
    available = facts["facts"].get("us-gaap", {}).keys()
    for tag in tags:
        rows.append({
            "ticker": ticker,
            "sector": companies[ticker]["sector"],
            "tag": tag,
            "present": tag in available,
        })

coverage = pd.DataFrame(rows)
coverage.pivot(index="ticker", columns="tag", values="present")

tag,NetCashProvidedByUsedInOperatingActivities,NetIncomeLoss,PaymentsToAcquirePropertyPlantAndEquipment,ResearchAndDevelopmentExpense,Revenues
ticker,,,,,
AAPL,True,True,True,True,True
CVX,True,True,False,True,True
JNJ,True,True,True,True,True
MSFT,True,True,True,True,True
NEE,True,True,False,False,True
NVDA,True,True,True,True,True
PFE,True,True,True,False,True
SYK,True,True,True,True,True


In [74]:
cik = get_cik["XOM"]
print("CIK used:", cik)

# Find every entry in the SEC file whose ticker is XOM, or whose title mentions Exxon
for entry in ticker_data.values():
    if entry["ticker"] == "XOM" or "EXXON" in entry["title"].upper():
        print(entry)

CIK used: 0002115436
{'cik_str': 2115436, 'ticker': 'XOM', 'title': 'ExxonMobil Holdings Corp'}
{'cik_str': 2014337, 'ticker': 'NPT', 'title': 'Texxon Holding Ltd'}


In [75]:
test_cik = "0000034088"
url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{test_cik}.json"
r = requests.get(url, headers=HEADERS)
print(r.status_code)

if r.status_code == 200:
    d = r.json()
    print(d["entityName"])
    print(list(d["facts"].keys()))
    print("Revenues present:", "Revenues" in d["facts"].get("us-gaap", {}))

200
Exxon Mobil Corporation
['dei', 'us-gaap', 'ecd']
Revenues present: True


In [76]:
# The SEC ticker file maps XOM to a recently created holding entity with no
# financial filing history. Point to the operating company's filer instead.
cik_overrides = {"XOM": "0000034088"}

In [77]:
for ticker in companies:
    print(ticker, get_cik[ticker], companies[ticker]["name"])

MSFT 0000789019 Microsoft
NVDA 0001045810 Nvidia
AAPL 0000320193 Apple
PFE 0000078003 Pfizer
JNJ 0000200406 Johnson & Johnson
SYK 0000310764 Stryker
XOM 0002115436 ExxonMobil
CVX 0000093410 Chevron
NEE 0000753308 NextEra Energy


In [78]:
print(type(get_cik))

<class 'dict'>


In [79]:
cik_overrides = {"XOM": "0000034088"}

def get_cik(ticker):
    return cik_overrides.get(ticker, ticker_to_cik[ticker])

In [80]:
print(get_cik("XOM"))   # should print 0000034088
print(get_cik("MSFT"))  # should print 0000789019

0000034088
0000789019


In [81]:
company_facts = {}

for ticker in companies:
    url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{get_cik(ticker)}.json"
    response = requests.get(url, headers=HEADERS)

    if response.status_code != 200:
        print(f"{ticker}: failed (status {response.status_code})")
    else:
        facts = response.json()
        if "us-gaap" in facts.get("facts", {}):
            company_facts[ticker] = facts
        else:
            print(f"{ticker}: no us-gaap facts")

    time.sleep(1)

print(f"\nLoaded {len(company_facts)} of {len(companies)} companies")


Loaded 9 of 9 companies


In [82]:
import pandas as pd

rows = [
    {
        "ticker": ticker,
        "tag": tag,
        "present": tag in facts["facts"]["us-gaap"],
    }
    for ticker, facts in company_facts.items()
    for tag in tags
]

coverage = pd.DataFrame(rows).pivot(index="ticker", columns="tag", values="present")
coverage

tag,NetCashProvidedByUsedInOperatingActivities,NetIncomeLoss,PaymentsToAcquirePropertyPlantAndEquipment,ResearchAndDevelopmentExpense,Revenues
ticker,,,,,
AAPL,True,True,True,True,True
CVX,True,True,False,True,True
JNJ,True,True,True,True,True
MSFT,True,True,True,True,True
NEE,True,True,False,False,True
NVDA,True,True,True,True,True
PFE,True,True,True,False,True
SYK,True,True,True,True,True
XOM,True,True,True,True,True


In [83]:
alt_keywords = {
    "GrossProfit": ["GrossProfit", "CostOfRevenue", "CostOfGoodsAndServicesSold", "CostOfGoodsSold"],
    "OperatingIncomeLoss": ["OperatingIncomeLoss", "IncomeLossFromContinuingOperationsBefore"],
    "PaymentsToAcquirePropertyPlantAndEquipment": ["PaymentsToAcquireProductiveAssets", "PaymentsForCapitalImprovements", "PaymentsToAcquirePropertyPlant", "CapitalExpenditure"],
    "ResearchAndDevelopmentExpense": ["ResearchAndDevelopmentExpense"],
}

for ticker, facts in company_facts.items():
    available = facts["facts"]["us-gaap"].keys()
    for tag in tags:
        if tag in available:
            continue
        candidates = [
            t for t in available
            if any(kw.lower() in t.lower() for kw in alt_keywords[tag])
        ]
        print(f"{ticker} / {tag}: {candidates if candidates else 'NO CANDIDATES'}")

PFE / ResearchAndDevelopmentExpense: ['ResearchAndDevelopmentExpenseExcludingAcquiredInProcessCost']
CVX / PaymentsToAcquirePropertyPlantAndEquipment: ['PaymentsToAcquireProductiveAssets']
NEE / ResearchAndDevelopmentExpense: NO CANDIDATES
NEE / PaymentsToAcquirePropertyPlantAndEquipment: ['CapitalExpendituresIncurredButNotYetPaid']


In [84]:
nee = company_facts["NEE"]["facts"]["us-gaap"].keys()
[t for t in nee if "Payments" in t or "PropertyPlantAndEquipment" in t]

['AccumulatedDepreciationDepletionAndAmortizationPropertyPlantAndEquipment',
 'DeferredTaxLiabilitiesPropertyPlantAndEquipment',
 'PaymentsForProceedsFromNuclearFuel',
 'PaymentsForProceedsFromOtherInvestingActivities',
 'PaymentsForRepurchaseOfCommonStock',
 'PaymentsOfDividends',
 'PaymentsOfStockIssuanceCosts',
 'PaymentsToAcquireOtherInvestments',
 'PaymentsToAcquireOtherLoansAndLeasesHeldForInvestment',
 'PaymentsToInvestInDecommissioningFund',
 'ProceedsFromPaymentsForOtherFinancingActivities',
 'PropertyPlantAndEquipmentFairValueDisclosure',
 'PropertyPlantAndEquipmentGross',
 'PropertyPlantAndEquipmentNet',
 'PropertyPlantAndEquipmentUsefulLifeMaximum',
 'PublicUtilitiesPropertyPlantAndEquipmentDisclosureOfCompositeDepreciationRateForPlantsInService',
 'PublicUtilitiesPropertyPlantAndEquipmentNet',
 'SalesTypeAndDirectFinancingLeasesLeaseReceivablePaymentsToBeReceived',
 'OperatingLeaseLeaseIncomeLeasePayments']

In [85]:
nee_facts = company_facts["NEE"]["facts"]
print(list(nee_facts.keys()))

# broader sweep of us-gaap for anything capex-shaped
[t for t in nee_facts["us-gaap"] if any(k in t for k in ["Capital", "Construction", "Additions", "Investing"])]

['dei', 'us-gaap', 'ffd']


['AdditionalPaidInCapitalCommonStock',
 'AdjustmentsToAdditionalPaidInCapitalStockIssuedIssuanceCosts',
 'CapitalExpendituresIncurredButNotYetPaid',
 'CapitalizationLongtermDebtAndEquity',
 'ConstructionInProgressGross',
 'ConstructionPayableCurrent',
 'EquityMethodInvestmentSummarizedFinancialInformationEquityOrCapital',
 'InterestCostsCapitalized',
 'InterestCostsIncurredCapitalized',
 'NetCashProvidedByUsedInInvestingActivities',
 'PaymentsForProceedsFromOtherInvestingActivities',
 'ProceedsFromAdvancesForConstruction',
 'ProceedsFromEquityMethodInvestmentDividendsOrDistributionsReturnOfCapital',
 'PublicUtilitiesAllowanceForFundsUsedDuringConstructionAdditions',
 'PublicUtilitiesAllowanceForFundsUsedDuringConstructionCapitalizedCostOfEquity',
 'PublicUtilitiesAllowanceForFundsUsedDuringConstructionRate']

In [86]:
candidates = ["DUK", "SO", "D", "COP", "SLB", "PSX", "VLO", "OXY"]

for tkr in candidates:
    cik = ticker_to_cik.get(tkr)
    if cik is None:
        print(f"{tkr}: not in ticker file")
        continue

    r = requests.get(f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json", headers=HEADERS)
    if r.status_code != 200:
        print(f"{tkr}: status {r.status_code}")
        time.sleep(1)
        continue

    d = r.json()
    gaap = d["facts"].get("us-gaap", {})
    if not gaap:
        print(f"{tkr}: no us-gaap ({list(d['facts'].keys())})")
        time.sleep(1)
        continue

    have = [t for t in tags if t in gaap]
    print(f"{tkr:5} {d['entityName'][:28]:30} {len(have)}/6  missing: {[t for t in tags if t not in gaap]}")
    time.sleep(1)

DUK   DUKE ENERGY CORPORATION        4/6  missing: ['ResearchAndDevelopmentExpense']
SO    SOUTHERN CO                    4/6  missing: ['ResearchAndDevelopmentExpense']
D     DOMINION ENERGY, INC           4/6  missing: ['ResearchAndDevelopmentExpense']
COP   CONOCOPHILLIPS                 4/6  missing: ['PaymentsToAcquirePropertyPlantAndEquipment']
SLB   SLB LIMITED/NV                 5/6  missing: []
PSX   Phillips 66                    3/6  missing: ['Revenues', 'PaymentsToAcquirePropertyPlantAndEquipment']
VLO   VALERO ENERGY CORP/TX          3/6  missing: ['Revenues', 'ResearchAndDevelopmentExpense']
OXY   OCCIDENTAL PETROLEUM CORPORA   4/6  missing: ['ResearchAndDevelopmentExpense']
